# Real-Time Image Animation

---

    -- By: Komal Shahid
    -- Due Date: July 14th , 2024
    -- DSC 630
    -- Milestone 3

---


In [1]:
import cv2
import mediapipe as mp
import pygame
from kaggle.api.kaggle_api_extended import KaggleApi
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from moviepy.editor import VideoClip
from transformers import AutoTokenizer, AutoModel
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

Matplotlib is building the font cache; this may take a moment.
Fontconfig warning: ignoring UTF-8: not a valid region tag
2024-07-14 23:58:25.405148: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


pygame 2.6.0 (SDL 2.28.4, Python 3.12.0)
Hello from the pygame community. https://www.pygame.org/contribute.html


ModuleNotFoundError: No module named 'moviepy'

In [1]:

class ImageAnimator:
    """
    Class for real-time image animation using deep learning.
    """
    def __init__(self, model_name):
        """
        Initialize the ImageAnimator with a specific model.

        Args:
        model_name : str
            The name of the pre-trained model to use for animation. 
            Model = zyand/animate-anything-v1.02  and this model has been called from Hugging Face .
        """
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands()
        self.cap = cv2.VideoCapture(0)

    def capture_hand_motion(self):
        """
        Capture hand motion using the webcam and process it with MediaPipe Hands.

        Returns:
        results : mediapipe.python.solution_base.SolutionOutputs
            The processed hand motion data.
        """
        success, image = self.cap.read()
        if not success:
            return None
        results = self.hands.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        return results

    def generate_art(self, features):
        """
        Generate 3D abstract art using the pre-trained model.

        Args:
        features : numpy.ndarray
            The features extracted from the hand motion data.

        Returns:
        art_3d : numpy.ndarray
            The generated 3D abstract art.
        """
        art = self.model.predict(features)
        art_3d = np.reshape(art, (art.shape[0], art.shape[1], 1))
        return art_3d

    def create_animation(self, make_frame, duration=4):
        """
        Create a video that mimics wave pattern.

        Args:
        make_frame : function
            A function that returns an image array for each frame.
        duration : int, optional
            The duration of the video in seconds (default is 4).

        Returns:
        animation : moviepy.video.VideoClip.VideoClip
            The created animation.
        """
        animation = VideoClip(make_frame, duration=duration)
        animation.write_videofile("my_animation.mp4", fps=24)
        return animation

    def download_dataset(self, dataset_name):
        """
        Download a dataset from Kaggle.

        Args:
        dataset_name : str
            The name of the dataset to download.
        """
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files(dataset_name)
    

     def create_3d_plot(self, x, y, z):
        """
        Create a 3D plot.

        Args:
        x : numpy.ndarray
            The x-coordinates for the 3D plot.
        y : numpy.ndarray
            The y-coordinates for the 3D plot.
        z : numpy.ndarray
            The z-coordinates for the 3D plot.

        Returns:
        fig : matplotlib.figure.Figure
            The created 3D plot.
        """
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        ax.plot_surface(x, y, z, color='c', alpha=0.6, linewidth=0)
        return fig

    def update_animation(self, fig, update_func, frames, interval):
        """
        Update the animation.

        Args:
        fig : matplotlib.figure.Figure
            The figure to update.
        update_func : function
            The function to use for updating the figure.
        frames : array_like
            The frames to pass to the update function.
        interval : int
            The delay between frames in milliseconds.

        Returns:
        ani : matplotlib.animation.FuncAnimation
            The updated animation.
        """
        ani = animation.FuncAnimation(fig, update_func, frames=frames, interval=interval)
        return ani

    def release_resources(self):
        """
        Release the resources used by the ImageAnimator.
        """
        self.cap.release()
    
    def make_frame(self, t):
        """
        Create a frame for the video.

        Args:
        t : float
            The time parameter for the wave pattern.

        Returns:
        numpy.ndarray
            An image array for the frame.
        """
        return np.zeros((480, 640, 3))

    def update(self, t):
        """
        Update the animation.

        Args:
        t : float
            The time parameter for the wave pattern.

        Returns:
        tuple
            The updated surface plot.
        """
        self.ax.clear()
        z = np.sin(np.sqrt(self.x**2 + self.y**2) - t)
        wave = self.ax.plot_surface(self.x, self.y, z, color='c', alpha=0.6, linewidth=0)
        return wave,

    def run(self):
        """
        Run the image animation.
        """
        # Create a video that mimics wave pattern
        animation = self.create_animation(self.make_frame)

        # Download the dataset
        self.download_dataset('bryanb/abstract-art-gallery')

        # Create the x, y, and z coordinate arrays
        self.x = np.linspace(-5, 5, 101)
        self.y = np.linspace(-5, 5, 101)
        self.x, self.y = np.meshgrid(self.x, self.y)
        z = np.sin(np.sqrt(self.x**2 + self.y**2))

        # Create a 3D plot
        fig = self.create_3d_plot(self.x, self.y, z)

        # Update the animation
        ani = self.update_animation(fig, self.update, frames=np.arange(0, 2*np.pi, 0.1), interval=100)

        # Release the resources
        self.release_resources()

        plt.show()


In [ ]:
# Initialize the ImageAnimator with the 'zyand/animate-anything-v1.02' model
animator = ImageAnimator("zyand/animate-anything-v1.02")

In [ ]:
# Run the image animation
animator.run()